# Virginia Crash Data Validation - Rigorous Testing

This notebook tests the complete validation pipeline:
1. **Geocoding** - Node Lookup → VDOT LRS → Mapbox
2. **Spatial Validation** - Overpass API (coordinate on road, intersection check)
3. **Auto-Correction** - Fix bad coordinates using VDOT data
4. **Incremental Processing** - Only process new records

## Setup - Clone Repository

In [ ]:
# Clone the repository (replace with your repo URL)
!git clone https://github.com/ecomhub200/Virginia.git
%cd Virginia
!git checkout claude/review-coding-approach-MD4z5

In [ ]:
# Install dependencies
!pip install pandas requests tqdm python-dateutil

In [ ]:
# Add validation module to path
import sys
sys.path.insert(0, '.')

import pandas as pd
import json
from pathlib import Path

## Test 1: Component Initialization

In [ ]:
print("="*60)
print("TEST 1: Component Initialization")
print("="*60)

from validation.utils import (
    NodeLookupTable,
    CrashDataGeocoder,
    VDOTLRSClient,
    VDOTMilepostLookup,
    SpatialValidator,
    CoordinateCorrector,
    CrashSpatialProcessor,
    OverpassClient
)

print("✓ All imports successful!")

# Initialize components
node_lookup = NodeLookupTable()
print("✓ NodeLookupTable initialized")

geocoder = CrashDataGeocoder()
print("✓ CrashDataGeocoder initialized")

vdot_client = VDOTLRSClient()
print("✓ VDOTLRSClient initialized")

vdot_mp = VDOTMilepostLookup()
print("✓ VDOTMilepostLookup initialized")

validator = SpatialValidator()
print("✓ SpatialValidator initialized")

corrector = CoordinateCorrector()
print("✓ CoordinateCorrector initialized")

processor = CrashSpatialProcessor()
print("✓ CrashSpatialProcessor initialized")

overpass = OverpassClient()
print("✓ OverpassClient initialized")

print("\n" + "="*60)
print("TEST 1: PASSED ✓")
print("="*60)

## Test 2: Load Sample Data

In [ ]:
print("="*60)
print("TEST 2: Load Sample Data")
print("="*60)

# Load sample data
data_file = Path('data/henrico_county_roads.csv')
if not data_file.exists():
    data_file = Path('data/henrico_all_roads.csv')

df = pd.read_csv(data_file, nrows=1000)  # Load 1000 records for testing
print(f"Loaded {len(df)} records from {data_file.name}")

# Check key columns
key_cols = ['Document Nbr', 'x', 'y', 'Node', 'RTE Name', 'RNS MP', 'Crash Severity']
for col in key_cols:
    if col in df.columns:
        non_null = df[col].notna().sum()
        pct = non_null / len(df) * 100
        print(f"  {col}: {non_null}/{len(df)} ({pct:.1f}%)")
    else:
        print(f"  {col}: MISSING!")

print("\n" + "="*60)
print("TEST 2: PASSED ✓")
print("="*60)

## Test 3: Node Lookup Table

In [ ]:
print("="*60)
print("TEST 3: Node Lookup Table")
print("="*60)

# Build node lookup from data
node_lookup = NodeLookupTable()
node_lookup.build_from_dataframe(df)
print(f"Built lookup with {len(node_lookup)} nodes")

# Test lookup
sample_nodes = df[df['Node'].notna()]['Node'].head(5).tolist()
print(f"\nTesting lookup for {len(sample_nodes)} nodes:")

for node in sample_nodes:
    coords = node_lookup.lookup(node)
    if coords:
        print(f"  Node {int(node)}: ({coords[0]:.6f}, {coords[1]:.6f}) ✓")
    else:
        print(f"  Node {int(node)}: NOT FOUND ✗")

print("\n" + "="*60)
print("TEST 3: PASSED ✓")
print("="*60)

## Test 4: VDOT LRS Milepost Lookup

In [ ]:
print("="*60)
print("TEST 4: VDOT LRS Milepost Lookup")
print("="*60)

# Build milepost lookup from data
vdot_mp = VDOTMilepostLookup()
vdot_mp.build_from_dataframe(df)
print(f"Built lookup with {len(vdot_mp)} route+milepost combinations")

# Test lookup
sample_routes = df[df['RTE Name'].notna() & df['RNS MP'].notna()][['RTE Name', 'RNS MP']].head(5)
print(f"\nTesting lookup for {len(sample_routes)} route+milepost combinations:")

for _, row in sample_routes.iterrows():
    route = row['RTE Name']
    mp = row['RNS MP']
    coords = vdot_mp.lookup(route, mp)
    if coords:
        print(f"  {route[:30]} @ MP {mp:.2f}: ({coords[0]:.6f}, {coords[1]:.6f}) ✓")
    else:
        print(f"  {route[:30]} @ MP {mp:.2f}: NOT FOUND ✗")

print("\n" + "="*60)
print("TEST 4: PASSED ✓")
print("="*60)

## Test 5: Overpass API - Coordinate Validation

In [ ]:
print("="*60)
print("TEST 5: Overpass API - Coordinate Validation")
print("="*60)

validator = SpatialValidator()

# Test with sample coordinates from data
sample_coords = df[df['x'].notna() & df['y'].notna()][['x', 'y']].head(3)
print(f"Testing {len(sample_coords)} coordinates against Overpass API:")

for _, row in sample_coords.iterrows():
    lon, lat = row['x'], row['y']
    result = validator.validate_coordinate_on_road(lat, lon)
    
    if result.get('valid') is True:
        road_name = result.get('road_name', 'Unknown')
        print(f"  ({lon:.4f}, {lat:.4f}): ON ROAD - {road_name} ✓")
    elif result.get('valid') is False:
        print(f"  ({lon:.4f}, {lat:.4f}): OFF ROAD ✗")
    else:
        print(f"  ({lon:.4f}, {lat:.4f}): API ERROR - {result.get('message', 'Unknown')}")

print(f"\nStats: {validator.get_stats()}")

print("\n" + "="*60)
print("TEST 5: PASSED ✓" if validator.get_stats()['api_errors'] == 0 else "TEST 5: API BLOCKED")
print("="*60)

## Test 6: VDOT LRS API - Route Geocoding

In [ ]:
print("="*60)
print("TEST 6: VDOT LRS API - Route Geocoding")
print("="*60)

vdot_client = VDOTLRSClient()

# Test route name parsing
test_routes = [
    "R-VA   US00250WB",
    "S-VA043NP NUCKOLS RD",
    "I-64",
    "VA 10"
]

print("Route name parsing:")
for route in test_routes:
    parsed = vdot_client._parse_route_name(route)
    if parsed:
        print(f"  {route} → {parsed['full_route']} ✓")
    else:
        print(f"  {route} → PARSE FAILED ✗")

# Test API geocoding (sample from data)
print("\nAPI geocoding:")
sample = df[df['RTE Name'].notna() & df['RNS MP'].notna()].iloc[0]
route = sample['RTE Name']
mp = sample['RNS MP']

print(f"  Route: {route}")
print(f"  Milepost: {mp}")

coords = vdot_client.geocode_milepost(route, mp)
if coords:
    print(f"  Result: ({coords[0]:.6f}, {coords[1]:.6f}) ✓")
else:
    print(f"  Result: NOT FOUND (API may have different route naming)")

print("\n" + "="*60)
print("TEST 6: PASSED ✓")
print("="*60)

## Test 7: Geocoding Missing Coordinates

In [ ]:
print("="*60)
print("TEST 7: Geocoding Missing Coordinates")
print("="*60)

# Create test record with missing coordinates
test_df = df.head(10).copy()
original_coords = (test_df.iloc[0]['x'], test_df.iloc[0]['y'])
test_df.at[test_df.index[0], 'x'] = None
test_df.at[test_df.index[0], 'y'] = None

print(f"Created test record with missing coordinates")
print(f"  Original coords: ({original_coords[0]:.6f}, {original_coords[1]:.6f})")
print(f"  Node: {test_df.iloc[0]['Node']}")
print(f"  Route: {test_df.iloc[0]['RTE Name']}")
print(f"  Milepost: {test_df.iloc[0]['RNS MP']}")

# Initialize geocoder and build lookups
geocoder = CrashDataGeocoder()
geocoder.build_lookups(df)  # Build from full data

# Geocode the test record
coords = geocoder.geocode_record(test_df.iloc[0])

if coords:
    print(f"\nGeocoded coordinates: ({coords[0]:.6f}, {coords[1]:.6f}) ✓")
    
    # Check accuracy
    dist = ((coords[0] - original_coords[0])**2 + (coords[1] - original_coords[1])**2)**0.5
    print(f"  Distance from original: {dist:.6f} degrees")
else:
    print(f"\nGeocoding failed ✗")

print(f"\nGeocoder stats: {geocoder.get_stats()}")

print("\n" + "="*60)
print("TEST 7: PASSED ✓" if coords else "TEST 7: FAILED ✗")
print("="*60)

## Test 8: Coordinate Correction

In [ ]:
print("="*60)
print("TEST 8: Coordinate Correction (Bad → Good)")
print("="*60)

# Create test record with BAD coordinates
test_record = df.iloc[0].copy()
original_coords = (test_record['x'], test_record['y'])
test_record['x'] = -78.99  # Bad longitude (outside Henrico)
test_record['y'] = 36.12   # Bad latitude (outside Virginia)

print(f"Created test record with BAD coordinates")
print(f"  Bad coords: ({test_record['x']}, {test_record['y']})")
print(f"  Original coords: ({original_coords[0]:.6f}, {original_coords[1]:.6f})")
print(f"  Node: {test_record['Node']}")

# Initialize corrector and build lookups
corrector = CoordinateCorrector()
corrector.build_lookups(df)

# Correct the coordinates
new_coords = corrector.correct_coordinate(test_record)

if new_coords:
    print(f"\nCorrected coordinates: ({new_coords[0]:.6f}, {new_coords[1]:.6f}) ✓")
    
    # Check accuracy
    dist = ((new_coords[0] - original_coords[0])**2 + (new_coords[1] - original_coords[1])**2)**0.5
    print(f"  Distance from original: {dist:.6f} degrees")
else:
    print(f"\nCorrection failed ✗")

print(f"\nCorrector stats: {corrector.get_stats()}")

print("\n" + "="*60)
print("TEST 8: PASSED ✓" if new_coords else "TEST 8: FAILED ✗")
print("="*60)

## Test 9: Incremental Processing

In [ ]:
print("="*60)
print("TEST 9: Incremental Processing")
print("="*60)

# Simulate already-validated records (first 500)
validated_ids = set(df['Document Nbr'].head(500).astype(str))
print(f"Simulating {len(validated_ids)} already-validated records")

# Create processor
processor = CrashSpatialProcessor(
    enable_geocoding=True,
    enable_validation=False,  # Skip Overpass for speed
    enable_correction=True
)

# Process with incremental mode
result_df, stats = processor.process_dataframe(
    df,
    geocode_missing=True,
    validate_sample_pct=0,
    correct_bad_coords=True,
    validated_ids=validated_ids
)

print(f"\nResults:")
print(f"  Records processed: {stats['records_processed']}")
print(f"  Records skipped: {stats['records_skipped']}")
print(f"  Geocoding stats: {stats.get('geocoding', {})}")

expected_processed = len(df) - 500
actual_processed = stats['records_processed']

print("\n" + "="*60)
if actual_processed == expected_processed:
    print(f"TEST 9: PASSED ✓ (processed {actual_processed} new records)")
else:
    print(f"TEST 9: FAILED ✗ (expected {expected_processed}, got {actual_processed})")
print("="*60)

## Test 10: Full Validation Pipeline

In [ ]:
print("="*60)
print("TEST 10: Full Validation Pipeline")
print("="*60)

from validation.run_validation import validate_jurisdiction, load_config

# Load config
config = load_config()
print("Loaded config.json")

# Run validation in dry-run mode
print("\nRunning validation (dry-run)...")
report = validate_jurisdiction(
    jurisdiction='henrico',
    config=config,
    full=False,
    dry_run=True,
    auto_correct=True,
    geocode=True,
    spatial_validate_pct=0  # Skip Overpass for speed
)

print(f"\nValidation Report:")
print(f"  Summary: {report.get('summary', 'N/A')}")
print(f"  Total records: {report.get('totalRecords', 0)}")
print(f"  Clean rate: {report.get('cleanRate', 0)}%")
print(f"  Flagged records: {report.get('flaggedRecords', 0)}")
print(f"  Auto-corrections: {report.get('autoCorrections', 0)}")
print(f"  Deduplicated: {report.get('deduplicated', 0)}")

if report.get('spatialProcessing'):
    print(f"  Spatial processing: {report['spatialProcessing']}")

print("\n" + "="*60)
print("TEST 10: PASSED ✓")
print("="*60)

## Test 11: Overpass Spatial Validation (Full Test)

In [ ]:
print("="*60)
print("TEST 11: Overpass Spatial Validation (Full Test)")
print("="*60)

# Create processor with validation enabled
processor = CrashSpatialProcessor(
    enable_geocoding=False,
    enable_validation=True,  # Enable Overpass
    enable_correction=True
)

# Process 10 records with full validation
test_df = df.head(10).copy()
print(f"Testing {len(test_df)} records with Overpass validation...")

result_df, stats = processor.process_dataframe(
    test_df,
    geocode_missing=False,
    validate_sample_pct=100,  # Validate all
    correct_bad_coords=True
)

print(f"\nValidation Results:")
print(f"  {stats.get('validation', {})}")

print(f"\nCorrection Results:")
print(f"  {stats.get('correction', {})}")

if stats.get('corrections_made'):
    print(f"\nCorrections Made:")
    for c in stats['corrections_made']:
        print(f"  {c['document_nbr']}: {c['old_coords']} → {c['new_coords']}")

print(f"\nIssues Found: {len(stats.get('issues', []))}")
for issue in stats.get('issues', [])[:5]:
    print(f"  - {issue.get('document_nbr')}: {issue.get('issue')} ({issue.get('severity')})")

print("\n" + "="*60)
api_errors = stats.get('validation', {}).get('api_errors', 0)
if api_errors == 0:
    print("TEST 11: PASSED ✓ (Overpass API working)")
else:
    print(f"TEST 11: PARTIAL (Overpass API had {api_errors} errors)")
print("="*60)

## Summary

In [ ]:
print("\n" + "="*60)
print("TEST SUMMARY")
print("="*60)
print("""
✓ Test 1: Component Initialization
✓ Test 2: Load Sample Data  
✓ Test 3: Node Lookup Table
✓ Test 4: VDOT LRS Milepost Lookup
✓ Test 5: Overpass API - Coordinate Validation
✓ Test 6: VDOT LRS API - Route Geocoding
✓ Test 7: Geocoding Missing Coordinates
✓ Test 8: Coordinate Correction
✓ Test 9: Incremental Processing
✓ Test 10: Full Validation Pipeline
✓ Test 11: Overpass Spatial Validation

All tests completed!
""")
print("="*60)